# Lantern — G0 evidence lab (2026-09-05)

Runs the G0 evidence-gathering checks from `docs/task-lantern-plan-v4-4.md` section 9.3,
narrowed per `docs/plan-amendments.md` amendment A2. Cells 1–7 and the W1/W2 write test
are adapted from the donor notebook `Silpo_scenario_lab4.ipynb`
(`BACKGROUND_MATERIALS.md`) — this notebook is a measuring instrument: it only reads
(33 read-only tools; the 6 write tools are blocked by an allowlist derived from the
server's own `readOnlyHint` annotations, per plan section 1.2). One deliberate exception:
the W2 cell performs a single scoped write-then-restore test with an explicit opt-in
flag, exactly as the donor notebook does, to verify write/read-back/idempotency
semantics (G0-09).

**Never run this in CI.** It needs your own OAuth session (phone+OTP login in your
browser) and writes evidence artifacts under `../docs/evidence/`.

| Cell | G0 item | Adapted from donor cell |
|---|---|---|
| Setup/imports | — | 1, 2 |
| Discovery | IV-01 egress regression | 3 |
| DCR + PKCE login + session | auth contract | 4, 5, 6 |
| G0-01 | `tools/list` snapshot + hash | 6 (extended) |
| Cart/slot snapshot | shared input for G0-02/05/08 | 7 |
| G0-02 | gap formula regression (narrowed per A2) | new |
| A7 | delivery-channel comparison (disclosure layer) | 11 (9b) |
| G0-08 | slot availability check (release 1.109.6) | new, reuses snapshot |
| G0-09 (W1) | write-tool schemas, read-only | 14 |
| G0-09 (W2) | write → read-back → idempotency, with restore | 15 |
| Save | `docs/evidence/g0-results.json` | 12 (adapted: local path, no Colab) |


In [ ]:
#@title 1. Setup
# Adapted from donor cell 1: LAB_DIR now points at the tracked evidence
# folder instead of a Google Drive path — this notebook runs locally, not
# in Colab.
MCP_URL = "https://mcp.silpo.ua/mcp"
REDIRECT_URI = "https://localhost/callback"
PROTOCOL_VERSION = "2025-06-18"
EVIDENCE_DIR = "../docs/evidence"
RUN_DATE = "2026-09-05"


In [ ]:
#@title 2. Imports and helpers
# Unchanged from donor cell 2.
import requests, json, secrets, hashlib, base64, urllib.parse, os, re
from datetime import datetime, timezone, timedelta

KYIV = timezone(timedelta(hours=3))
def now_kyiv(): return datetime.now(KYIV)

def parse_mcp_response(resp):
    ct = resp.headers.get("content-type", "")
    text = resp.text
    if "text/event-stream" in ct or text.lstrip().startswith(("event:", "data:")):
        datas = [l[5:].strip() for l in text.splitlines() if l.startswith("data:")]
        for d in reversed(datas):
            try: return json.loads(d)
            except Exception: pass
        raise ValueError("SSE without valid JSON: " + text[:300])
    return resp.json()

def pretty(x, n=1500):
    print(json.dumps(x, ensure_ascii=False, indent=2)[:n])
print("OK")


In [ ]:
#@title 3. Discovery — IV-01 egress regression (403 = network barrier, 401 = normal)
# Unchanged from donor cell 3. A 403 here means the network path to
# mcp.silpo.ua is blocked (plan section 8.1: proven only from a
# Ukrainian-IP host); a 401 is expected and normal before login.
probe = requests.post(MCP_URL, json={"jsonrpc":"2.0","id":1,"method":"initialize",
    "params":{"protocolVersion":PROTOCOL_VERSION,"capabilities":{},
              "clientInfo":{"name":"lantern-evidence-lab","version":"1.0"}}},
    headers={"Accept":"application/json, text/event-stream"}, timeout=20)
print("initialize without token:", probe.status_code, "(IV-01: expect 401, not 403)")

auth_meta = None
for path in ("/.well-known/oauth-protected-resource", "/.well-known/oauth-authorization-server"):
    r = requests.get("https://mcp.silpo.ua" + path, timeout=20)
    print(path, "->", r.status_code)
    if r.ok:
        m = r.json()
        if "authorization_endpoint" in m: auth_meta = m
        for srv in m.get("authorization_servers", []):
            r2 = requests.get(srv.rstrip("/") + "/.well-known/oauth-authorization-server", timeout=20)
            if r2.ok: auth_meta = r2.json()
assert auth_meta, "OAuth metadata not received — IV-01 FAIL, check network path"
AUTHZ, TOKEN = auth_meta["authorization_endpoint"], auth_meta["token_endpoint"]
REG = auth_meta.get("registration_endpoint")
print("\nIV-01 OK:", AUTHZ)


In [ ]:
#@title 4. Dynamic client registration (DCR)
# Unchanged from donor cell 4.
reg = requests.post(REG, json={
    "client_name": "lantern-evidence-lab",
    "redirect_uris": [REDIRECT_URI],
    "grant_types": ["authorization_code"],
    "response_types": ["code"],
    "token_endpoint_auth_method": "none"}, timeout=20)
reg.raise_for_status()
CLIENT_ID = reg.json()["client_id"]
print("client_id:", CLIENT_ID)


In [ ]:
#@title 5. Login (PKCE) — password is entered ONLY in your own browser
# Unchanged from donor cell 5. No credential ever passes through this
# notebook or this process.
verifier = base64.urlsafe_b64encode(secrets.token_bytes(48)).rstrip(b"=").decode()
challenge = base64.urlsafe_b64encode(hashlib.sha256(verifier.encode()).digest()).rstrip(b"=").decode()
params = {"response_type":"code","client_id":CLIENT_ID,"redirect_uri":REDIRECT_URI,
          "code_challenge":challenge,"code_challenge_method":"S256",
          "state":secrets.token_urlsafe(8)}
if auth_meta.get("scopes_supported"): params["scope"] = " ".join(auth_meta["scopes_supported"])
print("1) Open and log in:\n")
print(AUTHZ + "?" + urllib.parse.urlencode(params))
print("\n2) The browser redirects to https://localhost/callback?code=... (the page will not load — that is expected)")
print("3) Paste the FULL returned URL below.")
redirect_resp = input("\nReturn URL: ").strip()
code_val = urllib.parse.parse_qs(urllib.parse.urlparse(redirect_resp).query)["code"][0]
tok = requests.post(TOKEN, data={"grant_type":"authorization_code","code":code_val,
    "redirect_uri":REDIRECT_URI,"client_id":CLIENT_ID,"code_verifier":verifier}, timeout=20)
tok.raise_for_status()
ACCESS_TOKEN = tok.json()["access_token"]
print("Token held in session memory only. OK")


In [ ]:
#@title 6. MCP session + allowlist from server annotations
# Unchanged from donor cell 6. READ_ALLOWLIST is derived from the server's
# own readOnlyHint, not a hand-maintained list — plan section 9.1: "Pydantic-обгортка
# допустима як внутрішня boundary... unknown fields зберігаються, contract tests обов'язкові."
S = requests.Session()
S.headers.update({"Authorization": f"Bearer {ACCESS_TOKEN}",
                  "Accept": "application/json, text/event-stream",
                  "MCP-Protocol-Version": PROTOCOL_VERSION})
r = S.post(MCP_URL, json={"jsonrpc":"2.0","id":1,"method":"initialize",
    "params":{"protocolVersion":PROTOCOL_VERSION,"capabilities":{},
              "clientInfo":{"name":"lantern-evidence-lab","version":"1.0"}}}, timeout=30)
r.raise_for_status()
init = parse_mcp_response(r)
sid = r.headers.get("mcp-session-id")
if sid: S.headers["mcp-session-id"] = sid
S.post(MCP_URL, json={"jsonrpc":"2.0","method":"notifications/initialized"}, timeout=15)

r = S.post(MCP_URL, json={"jsonrpc":"2.0","id":2,"method":"tools/list","params":{}}, timeout=30)
r.raise_for_status()
tools = parse_mcp_response(r)["result"]["tools"]
TOOLS_RAW = json.dumps(tools, ensure_ascii=False, indent=2)

READ_ALLOWLIST = {t["name"] for t in tools if t.get("annotations", {}).get("readOnlyHint") is True}
WRITE_TOOLS = {t["name"] for t in tools if not t.get("annotations", {}).get("readOnlyHint")}
existing = {t["name"] for t in tools}
print(f"tools: {len(tools)} | read-only: {len(READ_ALLOWLIST)} | write (blocked here): {len(WRITE_TOOLS)}")

samples = {}
def call(name, args=None, _id=[100]):
    assert name in READ_ALLOWLIST, f"BLOCKED (not read-only): {name}"
    if name not in existing: return {"isError": True, "skipped": True}
    _id[0] += 1
    rr = S.post(MCP_URL, json={"jsonrpc":"2.0","id":_id[0],"method":"tools/call",
        "params":{"name":name,"arguments":args or {}}}, timeout=60)
    rr.raise_for_status()
    out = parse_mcp_response(rr)
    return out.get("result", out)

def rec(key, tool, args=None, quiet=False):
    res = call(tool, args or {})
    samples[key] = res
    err = bool(res.get("isError"))
    if not quiet: print(("FAIL " if err else "OK   ") + f"{key:20} <- {tool}")
    if err and not quiet: print("   ", json.dumps(res, ensure_ascii=False)[:200])
    return None if err else (res.get("structuredContent") or {})


## G0-01 — `tools/list` snapshot and hash

Plan section 9.3: "Restart/Run All ноутбука; snapshot схем і hash. JSON snapshot; diff
від серпневого; count — не KPI." The hash is what a schema-drift regression (plan
section 9.4) compares on the next run, not the raw tool count.

In [ ]:
#@title G0-01. tools/list snapshot + hash -> docs/evidence/
import hashlib, os

os.makedirs(EVIDENCE_DIR, exist_ok=True)
schema_hash = hashlib.sha256(TOOLS_RAW.encode("utf-8")).hexdigest()
snapshot_path = f"{EVIDENCE_DIR}/tools-list-{RUN_DATE}.json"
with open(snapshot_path, "w", encoding="utf-8") as f:
    json.dump({
        "captured_at": now_kyiv().isoformat(),
        "protocol_version": PROTOCOL_VERSION,
        "schema_hash": schema_hash,
        "tool_count": len(tools),
        "read_only_count": len(READ_ALLOWLIST),
        "write_count": len(WRITE_TOOLS),
        "tools": tools,
    }, f, ensure_ascii=False, indent=2)
print(f"G0-01: wrote {snapshot_path}")
print(f"schema_hash: {schema_hash}")
print(f"tools: {len(tools)} total, {len(READ_ALLOWLIST)} read-only, {len(WRITE_TOOLS)} write")
print("Compare schema_hash against the previous run's evidence file, if one exists,")
print("to detect a release-channel schema drift (plan section 9.4).")


## Cart/slot snapshot

Shared input for G0-02, A7, and G0-08 below. Adapted from donor cell 7 ("Знімок «ДО»")
— the before/after diff machinery from the donor's scenario-comparison workflow is
dropped here; G0 only needs one honest snapshot of the current cart state, not a
before/after pair.

In [ ]:
#@title 7. Cart and slot snapshot
def snapshot():
    out = {}
    ref = rec("cart_ref", "silpo_get_my_shopping_cart", quiet=True)
    cid = (ref or {}).get("shoppingCartId")
    cart = rec("cart", "silpo_get_shopping_cart_by_id", {"shoppingCartId": cid}, quiet=True) if cid else {}
    c = (cart or {}).get("cart") or {}
    calc = c.get("calculation") or {}
    shp = (c.get("shipments") or [{}])[0]
    out["cart_id"] = cid
    out["branchId"] = shp.get("branchId")
    out["companyId"] = shp.get("companyId")
    out["deliveryType"] = c.get("deliveryType")
    out["timeslot"] = c.get("timeslot") or {}
    out["address"] = {k: (c.get("address") or {}).get(k) for k in
                      ("addressType","city","street","houseNumber","latitude","longitude")}
    out["totals"] = {k: calc.get(k) for k in
                     ("subTotal","subDiscount","total","totalAfterDiscounts","productsTotal")}
    out["validations"] = calc.get("validations") or []
    out["loyalty"] = (cart or {}).get("loyalty")
    out["checkout_link"] = "checkoutWebLink" in (cart or {})
    out["products"] = [{k: p.get(k) for k in
                        ("productId","name","slug","quantity","price","subDiscount","stock","companyId")}
                       for sh in (c.get("shipments") or []) for p in (sh.get("products") or [])]
    if out["branchId"]:
        sl = rec("slots", "silpo_get_time_slots",
                 {"branchId": out["branchId"], "deliveryTypes": [out["deliveryType"]], "limit": 50}, quiet=True)
        slots = (sl or {}).get("slots") or []
        out["slots_total"] = len(slots)
        out["slots_free"] = sum(1 for s in slots if s["available"])
        out["minOrderCost"] = sorted({s["minOrderCost"] for s in slots}) if slots else []
        out["deliveryCost"] = sorted({s["deliveryCost"] for s in slots}) if slots else []
        out["deliveryCostMap"] = slots[0]["deliveryCostMap"] if slots else []
        out["constraints"] = slots[0]["constraints"] if slots else {}
        ts = out["timeslot"]
        out["cart_slot_available"] = any(
            s["start"] == ts.get("start") and s["end"] == ts.get("end") and s["available"] for s in slots)
    out["restrictions"] = [r["slug"] for r in
                           (rec("restrictions", "silpo_get_my_food_restrictions", quiet=True) or {}).get("restrictions", [])]
    return out

cart_state = snapshot()
print("=" * 78)
print(f"{cart_state['address'].get('city')} | {cart_state['deliveryType']} | branch {str(cart_state['branchId'])[:8]}...")
print(f"items: {len(cart_state['products'])} | payable: {cart_state['totals']['totalAfterDiscounts']}")
print(f"minOrderCost: {cart_state.get('minOrderCost')} | delivery: {cart_state.get('deliveryCost')}")
print(f"slots: {cart_state.get('slots_total')} (free {cart_state.get('slots_free')}) | cart slot available: {cart_state.get('cart_slot_available')}")
print(f"checkoutWebLink: {cart_state['checkout_link']}")
print("\nvalidations:")
for v in cart_state["validations"]:
    print(f"  [{v['level']}] {v['message']} {json.dumps(v.get('context'), ensure_ascii=False)}")


## G0-02 — Gap formula regression (narrowed per amendment A2)

Per `docs/plan-amendments.md` A2: the plan's own field evidence already closed the
594.72 vs 599 anomaly (`mcp-field-capability-report.md` section 12.4 —
"[CLOSED] Anomaly explained": that value was `totalAfterDiscounts`, not
`productsTotal`). This cell is a plain regression check, not an investigation:
confirm `minOrderCost` compares against `productsTotal` on the cart(s) available
tonight, and flag anything within the DR-03 epsilon band as borderline rather than
auto-fixing it.

In [ ]:
#@title G0-02. Gap formula regression
EPSILON = 5.0  # DR-03 borderline band, in the cart's currency minor units-agnostic form

def check_gap(cart_state):
    calc = cart_state["totals"]
    products_total = calc.get("productsTotal")
    min_order_costs = cart_state.get("minOrderCost") or []
    if products_total is None or not min_order_costs:
        print("SKIP: missing productsTotal or minOrderCost — cannot check this cart")
        return
    for threshold in min_order_costs:
        gap = round(threshold - products_total, 2)
        has_code = any(v["message"] == "order.cost.min" for v in cart_state["validations"])
        if gap > EPSILON:
            expected = "error expected"
        elif gap > 0:
            expected = "borderline (epsilon band) — DR-03: do not auto-fix"
        else:
            expected = "should clear"
        print(f"threshold {threshold}: productsTotal {products_total} -> gap {gap} "
              f"[{expected}] | order.cost.min present: {has_code}")

check_gap(cart_state)
print("\nRepeat this cell against 2-3 different carts tonight if time allows "
      "(plan section 9.3, G0-02: \"Повторити перевірку на 3 кошиках\")")


## A7 — Delivery-channel comparison (disclosure layer)

Adapted from donor cell 11 ("9b. Порівняння типів доставки"). This is the read-only
evidence behind amendment A7: the same cart can be blocked under one delivery channel
and already clear under another, at zero extra cost. Uses `cart_state` from the
snapshot above instead of the donor's `after` variable (no before/after pair needed
here).

In [ ]:
#@title A7. Delivery-channel comparison
# Gap is computed from productsTotal, never totalAfterDiscounts — see DR-03 and
# amendment A2: comparing against the post-bonus total gave a false verdict in the
# donor's own field testing (an available channel misread as blocked).
a = cart_state["address"]
T = cart_state["totals"] or {}
payable = T.get("totalAfterDiscounts") or 0
goods = T.get("productsTotal") or 0
print(f"{a.get('city')} | goods: {goods} | payable: {payable} | current type: {cart_state['deliveryType']}\n")

dt = call("silpo_get_available_delivery_types",
          {"latitude": float(a["latitude"]), "longitude": float(a["longitude"])})
samples["delivery_types"] = dt
options = ((dt or {}).get("structuredContent") or {}).get("options") or []
print(f"delivery types for these coordinates: {len(options)}")

def num_min(vals):
    xs = [v for v in vals if isinstance(v, (int, float))]
    return min(xs) if xs else None

rows = []
for o in options:
    name = o["deliveryType"]
    bid, guessed = o.get("branchId"), False
    if not bid:
        flag = "hasPickup" if name == "SelfPickup" else "hasNP"
        br = call("silpo_list_branches", {flag: True, "limit": 100})
        samples[f"branches_{name}"] = br
        blist = ((br or {}).get("structuredContent") or {}).get("branches") or []
        local = [b for b in blist
                 if (b.get("city") or "").strip().lower() == (a.get("city") or "").lower()]
        if local:
            bid = local[0]["branchId"]
        else:
            bid, guessed = cart_state["branchId"], True

    r = call("silpo_get_time_slots", {"branchId": bid, "deliveryTypes": [name], "limit": 50})
    samples[f"slots_{name}"] = r
    slots = ((r or {}).get("structuredContent") or {}).get("slots") or []
    mn, dc = num_min([s.get("minOrderCost") for s in slots]), num_min([s.get("deliveryCost") for s in slots])
    rows.append({"type": name, "branchId": bid, "branch_guessed": guessed,
                 "slots": len(slots), "free": sum(1 for s in slots if s.get("available")),
                 "min": mn, "cost": dc,
                 "gap": None if mn is None else round(mn - goods, 2),
                 "map": (slots[0].get("deliveryCostMap") if slots else []) or []})

print(f"\n{'type':14} {'slots':>7} {'free':>6} {'min':>7} {'delivery':>8}  verdict")
print("-" * 80)
for x in rows:
    if not x["slots"]:      v = "no slots returned"
    elif x["gap"] > 0:      v = f"needs {x['gap']} more"
    elif not x["free"]:     v = "clears the sum, 0 free slots"
    else:                   v = "AVAILABLE NOW"
    if x["branch_guessed"]: v += "  (branch guessed from cart)"
    cost = "-" if x["cost"] is None else x["cost"]
    print(f"{x['type'][:14]:14} {x['slots']:>7} {x['free']:>6} "
          f"{str(x['min']):>7} {str(cost):>8}  {v}")

avail = [x for x in rows if x["free"] and x["gap"] is not None and x["gap"] <= 0]
cur = next((x for x in rows if x["type"] == cart_state["deliveryType"]), None)
if avail:
    best = min(avail, key=lambda x: (x["cost"] is not None, x["cost"] or 0))
    print(f"\nCHEAPEST NOW: {best['type']} - "
          f"{'no delivery fee' if best['cost'] is None else str(best['cost']) + ' fee'}")
    if cur and isinstance(cur["cost"], (int, float)):
        saving = cur["cost"] - (best["cost"] or 0)
        if saving > 0:
            print(f"savings vs {cur['type']}: {round(saving, 2)}, with zero extra purchase")

samples["delivery_compare"] = {"structuredContent": {"rows": rows,
                               "payable": payable, "productsTotal": goods,
                               "gap_base": "productsTotal"}}


## G0-08 — Slot behavior on release 1.109.6

`Discord.txt`: release-1.109.6 added a datetime normalizer for `get_time_slots`.
Plan section 9.3: "Повторити слотові прогони; Europe/Kyiv." Reuses the slot data
already fetched in the cart snapshot above; DR-02 requires the UTC-to-Kyiv conversion
to be checked by hand once here, since the domain code implementing it does not exist
yet at Stage 0.

In [ ]:
#@title G0-08. Slot behavior check
from zoneinfo import ZoneInfo

ts = cart_state.get("timeslot") or {}
if ts.get("start"):
    utc_start = datetime.fromisoformat(ts["start"])
    kyiv_start = utc_start.astimezone(ZoneInfo("Europe/Kyiv"))
    print(f"cart timeslot.start (raw):  {ts['start']}")
    print(f"cart timeslot.start (Kyiv): {kyiv_start.isoformat()}")
    print(f"cart_slot_available:        {cart_state.get('cart_slot_available')}")
else:
    print("No timeslot on the current cart — nothing to check this run.")

print(f"\nslots_total: {cart_state.get('slots_total')} | slots_free: {cart_state.get('slots_free')}")
print("constraints on this slot:", cart_state.get("constraints"))
print("\nRecord: does behavior match plan DR-02 (UTC internally, Kyiv on display)?")
print("and does the 1.109.6 datetime-normalizer release change anything vs. the")
print("field report's 8.3 observation (06:30 UTC == 09:30 Kyiv)?")


## G0-09 (W1) — Write-tool schemas, read-only

Unchanged from donor cell 14. Prints `inputSchema` and server annotations for every
write tool without calling any of them — the allowlist enforcement lives in the `call()`
helper defined in cell 6, which raises on any non-read-only tool name.

In [ ]:
#@title G0-09 (W1). Write-tool schemas, read-only
by_name = {t["name"] for t in tools} and {t["name"]: t for t in tools}
WRITE = sorted(WRITE_TOOLS)

print("=" * 78)
print("SERVER DECLARATIONS")
print("=" * 78)
print(f"{'tool':42} {'readOnly':>9} {'idempotent':>11} {'destructive':>12}")
print("-" * 78)
for n in WRITE:
    a = (by_name.get(n) or {}).get("annotations") or {}
    print(f"{n[:40]:42} {str(a.get('readOnlyHint')):>9} "
          f"{str(a.get('idempotentHint')):>11} {str(a.get('destructiveHint')):>12}")

print("\n" + "=" * 78)
print("ARGUMENT SCHEMAS")
print("=" * 78)
for n in WRITE:
    t = by_name.get(n) or {}
    sch = t.get("inputSchema") or {}
    props = sch.get("properties") or {}
    req = sch.get("required") or []
    print(f"\n### {n}")
    print(f"required: {req}")
    for k, v in props.items():
        mark = "*" if k in req else " "
        print(f"  {mark} {k:24} {str(v.get('type'))[:60]}")


## G0-09 (W2) — Write -> read-back -> idempotency, with restore

Unchanged from donor cell 15, except the flag defaults to **False** here — a live
write against a real cart is a deliberate, budget-and-state-aware action, handed over
to the user rather than defaulted on. Set `WRITE_ENABLED = True` and re-run only when
ready to perform one scoped, self-restoring write test.

In [ ]:
#@title G0-09 (W2). Live write -> read-back -> idempotency test (restores state)
import time

WRITE_ENABLED = False  # set True deliberately before running this cell

ALLOWED_WRITE = {"silpo_add_or_update_cart_products", "silpo_remove_cart_products"}
FORBIDDEN = {"silpo_clear_shopping_cart"}

tool = next(t for t in tools if t["name"] == "silpo_add_or_update_cart_products")
print("FULL SCHEMA silpo_add_or_update_cart_products")
print(json.dumps(tool.get("inputSchema"), ensure_ascii=False, indent=2))

if not WRITE_ENABLED:
    print("\n" + "=" * 78)
    print("WRITE_ENABLED = False - nothing executed (this is normal, not an error).")
    print("Schema above has been read. Set the flag and re-run when ready.")
else:
    def wcall(name, args, _id=[900]):
        assert name not in FORBIDDEN, f"PERMANENTLY FORBIDDEN: {name}"
        assert name in ALLOWED_WRITE, f"outside this test's allowlist: {name}"
        _id[0] += 1
        t0 = time.time()
        rr = S.post(MCP_URL, json={"jsonrpc": "2.0", "id": _id[0], "method": "tools/call",
                    "params": {"name": name, "arguments": args}}, timeout=60)
        rr.raise_for_status()
        out = parse_mcp_response(rr).get("result", {})
        print(f"   elapsed {time.time()-t0:.2f}s | isError={out.get('isError')}")
        return out

    def read_cart():
        r = call("silpo_get_shopping_cart_by_id", {"shoppingCartId": cart_state["cart_id"]})
        c = ((r or {}).get("structuredContent") or {}).get("cart") or {}
        calc = c.get("calculation") or {}
        prods = [p for sh in (c.get("shipments") or []) for p in (sh.get("products") or [])]
        return {
            "productsTotal": calc.get("productsTotal"),
            "checkout": "checkoutWebLink" in ((r or {}).get("structuredContent") or {}),
            "codes": [v["message"] for v in (calc.get("validations") or [])],
            "qty": {p["productId"]: p["quantity"] for p in prods},
        }

    cands = [p for p in cart_state["products"]
             if isinstance(p.get("stock"), (int, float)) and p["stock"] >= p["quantity"] + 1
             and (p.get("price") or 0) > 0]
    assert cands, "no suitable line item — need a product with stock above its cart quantity"
    tgt = min(cands, key=lambda p: p["price"])
    q0 = tgt["quantity"]
    print(f"\nTARGET: {tgt['name'][:60]} | productId {tgt['productId']} | "
          f"price {tgt['price']} | quantity {q0} -> {q0+1} -> back to {q0}")

    def args_for(q):
        return {"shoppingCartId": cart_state["cart_id"],
                "products": [{"productId": tgt["productId"],
                              "companyId": tgt["companyId"],
                              "branchId": cart_state["branchId"],
                              "quantity": q,
                              "addQuantity": False}]}

    s0 = read_cart()
    print(f"\nSTATE 0 (before write): products {s0['productsTotal']} | "
          f"target qty {s0['qty'].get(tgt['productId'])} | checkout {s0['checkout']}")

    r1 = wcall("silpo_add_or_update_cart_products", args_for(q0 + 1))
    s1 = read_cart()
    d1 = round((s1["productsTotal"] or 0) - (s0["productsTotal"] or 0), 2)
    print(f"READ-BACK 1: products {s1['productsTotal']} (delta {d1:+}) | "
          f"expected delta = unit price {tgt['price']}: "
          f"{'match' if abs(d1 - tgt['price']) < 0.02 else 'MISMATCH'}")

    r2 = wcall("silpo_add_or_update_cart_products", args_for(q0 + 1))
    s2 = read_cart()
    qa = s2["qty"].get(tgt["productId"])
    verdict = ("IDEMPOTENT: quantity sets the value" if qa == q0 + 1
               else "NOT idempotent: quantity adds" if qa == q0 + 2
               else f"unexpected: quantity became {qa}")
    print(f"READ-BACK 2 (same call repeated): products {s2['productsTotal']} | qty {qa} -> {verdict}")

    if qa == q0 + 1:
        wcall("silpo_add_or_update_cart_products", args_for(q0))
    else:
        wcall("silpo_remove_cart_products",
              {"shoppingCartId": cart_state["cart_id"],
               "products": [{"productId": tgt["productId"]}]})
        wcall("silpo_add_or_update_cart_products", args_for(q0))

    s3 = read_cart()
    ok = (s3["qty"].get(tgt["productId"]) == q0
          and abs((s3["productsTotal"] or 0) - (s0["productsTotal"] or 0)) < 0.02)
    print(f"READ-BACK 3 (restore): products {s3['productsTotal']} | qty {s3['qty'].get(tgt['productId'])} "
          f"-> {'restored' if ok else 'MISMATCH — fix by hand in the app'}")

    samples["write_test"] = {"structuredContent": {
        "target": {k: tgt.get(k) for k in ("productId", "name", "price", "quantity", "stock")},
        "delta_after_write": d1, "idempotent_verdict": verdict, "restored": ok,
        "declared_idempotent": (tool.get("annotations") or {}).get("idempotentHint")}}


## Save — `docs/evidence/g0-results.json`

Adapted from donor cell 12: no Google Drive mount logic (this runs locally), writes
directly under the tracked `docs/evidence/` folder. Raw MCP responses (`samples`) are
written to a file the `.gitignore` keeps local — only the summarized `g0-results.json`
is meant to be committed.

In [ ]:
#@title Save G0 results
os.makedirs(EVIDENCE_DIR, exist_ok=True)
results = {
    "run_date": RUN_DATE,
    "finished_at": now_kyiv().isoformat(),
    "cart_state": cart_state,
    "gap_check": "see G0-02 cell output above",
    "delivery_compare": (samples.get("delivery_compare") or {}).get("structuredContent"),
    "write_test": (samples.get("write_test") or {}).get("structuredContent"),
    "tools_count": len(tools),
    "read_only_tools": sorted(READ_ALLOWLIST),
    "write_tools_blocked": sorted(WRITE_TOOLS),
}
with open(f"{EVIDENCE_DIR}/g0-results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# Raw responses stay local (gitignored) — they can carry PII from a live account.
os.makedirs(f"{EVIDENCE_DIR}/raw", exist_ok=True)
with open(f"{EVIDENCE_DIR}/raw/g0-{RUN_DATE}_raw.json", "w", encoding="utf-8") as f:
    json.dump(samples, f, ensure_ascii=False, indent=2)

print(f"Wrote {EVIDENCE_DIR}/g0-results.json (tracked)")
print(f"Wrote {EVIDENCE_DIR}/raw/g0-{RUN_DATE}_raw.json (local only, gitignored)")
